# Chapter 26 — Optimisation

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch26_optimisation.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 26.1 Minimising a function

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize


def profit(q):
    q = np.asarray(q, dtype=float)
    return (60 - q) * q - (4 * q ** 3 / 30 - 3 * q ** 2 + 40 * q)


neg_profit = lambda q: -profit(q[0])          # minimize() works with arrays

for start in (1.0, 8.0):
    res = optimize.minimize(neg_profit, x0=[start], method="BFGS")
    print(f"start q = {start:>4}:  q* = {res.x[0]:6.3f}, "
          f"profit = {-res.fun:7.2f}, converged = {res.success}")

In [ ]:
def profit_threshold(q):
    return profit(q) - 120 * (1 / (1 + np.exp(-4 * (q - 10))))  # smooth step


neg2 = lambda q: -profit_threshold(q[0])
for start in (3.0, 7.0, 15.0):
    res = optimize.minimize(neg2, x0=[start], method="BFGS")
    print(f"start q = {start:>4}:  q* = {res.x[0]:6.3f}, "
          f"profit = {-res.fun:7.2f}")

grid = np.linspace(0, 20, 2001)
best = grid[np.argmax(profit_threshold(grid))]
print(f"grid search over [0, 20]: q* = {best:.2f}, "
      f"profit = {profit_threshold(best):.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.3))
ax.plot(grid, profit(grid), color="#9db7d5", linewidth=2,
        label="profit, no threshold")
ax.plot(grid, profit_threshold(grid), color="#1f4e79", linewidth=2,
        label="profit with fee above q = 10")
q_loc = optimize.minimize(neg2, x0=[15.0], method="BFGS").x[0]
ax.plot(q_loc, profit_threshold(q_loc), "o", color="#c55a11", markersize=8)
ax.annotate("local maximum\n(BFGS from q = 3 or 15)",
            (q_loc, profit_threshold(q_loc)), xytext=(11.3, 45), fontsize=9,
            color="#c55a11", arrowprops=dict(arrowstyle="-", color="grey"))
ax.plot(best, profit_threshold(best), "o", color="#1f4e79", markersize=8)
ax.annotate("global maximum\n(grid search)", (best, profit_threshold(best)),
            xytext=(3.0, 270), fontsize=9, color="#1f4e79",
            arrowprops=dict(arrowstyle="-", color="grey"))
ax.set_ylim(-50, 330)
ax.set_xlabel("output q")
ax.set_ylabel("profit")
ax.set_title("A local optimum is not always the global one", loc="left",
             fontsize=12)
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "Illustrative cost and demand functions.",
         fontsize=8, color="grey")
plt.show()

## 26.2 The consumer's problem with India's budget shares

In [ ]:
from pathlib import Path

import requests
from pypdf import PdfReader

PR_URL = ("https://mospi.gov.in/sites/default/files/press_release/"
          "cpi_pr_12feb15t.pdf")
pdf_path = Path("data/raw/cpi_pr_12feb15t.pdf")
if not pdf_path.exists():
    reply = requests.get(PR_URL, timeout=90, headers={
        "User-Agent": "python-for-indian-economics (teaching use)"})
    reply.raise_for_status()
    pdf_path.write_bytes(reply.content)

weights = {}
for line in PdfReader(pdf_path).pages[1].extract_text().splitlines():
    parts = line.split()
    if parts and parts[0] in "123456" and len(parts) > 10:
        weights[" ".join(parts[1:-9])] = float(parts[-3])   # combined weight
alpha = pd.Series(weights) / 100
alpha.round(4)

In [ ]:
def solve_consumer(alpha, p, m):
    a, p = np.asarray(alpha), np.asarray(p)
    utility = lambda x: -(a * np.log(x)).sum()               # minimise -U
    grad = lambda x: -a / x                                   # its gradient
    budget = {"type": "eq", "fun": lambda x: m - p @ x,
              "jac": lambda x: -p}
    x0 = np.full(len(a), m / p.sum())                         # spend equally
    return optimize.minimize(utility, x0, jac=grad, method="SLSQP",
                             constraints=[budget],
                             bounds=[(1e-9, None)] * len(a),
                             options={"ftol": 1e-12})


p0 = np.ones(len(alpha))
m = 100.0
res = solve_consumer(alpha, p0, m)
compare = pd.DataFrame({"numerical x*": res.x,
                        "analytic x* = alpha m / p": alpha * m / p0},
                       index=alpha.index)
print("converged:", res.success, "| iterations:", res.nit)
compare.round(4)

### The Lagrange multiplier is the marginal utility of income

In [ ]:
lam_foc = (alpha.to_numpy() / (p0 * res.x)).mean()
V = lambda income: -solve_consumer(alpha, p0, income).fun
h = 1e-3
lam_envelope = (V(m + h) - V(m - h)) / (2 * h)
print(f"lambda from first-order conditions: {lam_foc:.6f}")
print(f"lambda from dV/dm (envelope):       {lam_envelope:.6f}")
print(f"lambda from the formula 1/m:        {1 / m:.6f}")

## 26.3 How much has the cost of living risen? CES preferences

In [ ]:
from indiadata import mospi_cpi_groups

grp = mospi_cpi_groups("combined").drop_duplicates(["date", "group", "subgroup"])
grp = grp[grp["subgroup"].str.endswith("-Overall")]
latest = grp["date"].max()
now = grp[grp["date"] == latest].set_index("group")["index"]
general_published = now["General"]
names = {n.lower(): n for n in now.index}
p1 = np.array([now[names[k.lower()]] for k in alpha.index]) / 100
del grp
print(f"prices in {latest:%B %Y} (2012 = 1):")
print(pd.Series(p1, index=alpha.index).round(3).to_string())

In [ ]:
def ces_utility(x, a, sigma):
    if np.isclose(sigma, 1):
        return np.exp((a * np.log(x)).sum())
    r = (sigma - 1) / sigma
    return ((a ** (1 / sigma)) * x ** r).sum() ** (1 / r)


def cost_of_living_numeric(a, p, sigma):
    """Minimum spending at prices p to reach the utility of income 1 at p = 1."""
    u_bar = ces_utility(a, a, sigma)               # base bundle x = alpha
    cons = {"type": "ineq", "fun": lambda x: ces_utility(x, a, sigma) / u_bar - 1}
    res = optimize.minimize(lambda x: p @ x, x0=a, method="SLSQP",
                            constraints=[cons], bounds=[(1e-9, None)] * len(a),
                            options={"ftol": 1e-12, "maxiter": 500})
    return res.fun, res.x


def cost_of_living_formula(a, p, sigma):
    if np.isclose(sigma, 1):
        return np.prod(p ** a)
    return (a * p ** (1 - sigma)).sum() ** (1 / (1 - sigma))


a = alpha.to_numpy()
rows = []
for sigma in (0.25, 0.5, 1.0, 2.0):
    num, x_star = cost_of_living_numeric(a, p1, sigma)
    rows.append({"sigma": sigma, "numerical": num,
                 "formula": cost_of_living_formula(a, p1, sigma),
                 "food share at new prices": p1[0] * x_star[0] / num})
col = pd.DataFrame(rows).set_index("sigma")
laspeyres = a @ p1
print(f"Laspeyres (fixed basket) index: {100 * laspeyres:.1f}   "
      f"published CPI General, {latest:%b %Y}: {general_published:.1f}")
(100 * col[["numerical", "formula"]]).round(2).join(
    col["food share at new prices"].round(4))

## 26.4 Linear programming: the least-cost nutritious diet

In [ ]:
import re


def doca_retail_prices(refresh=False):
    """All-India average retail prices (Rs/kg) from the DoCA front page.

    Cached as data/raw/doca_retail_prices_YYYY-MM-DD.csv (the price date).
    """
    cached = sorted(Path("data/raw").glob("doca_retail_prices_*.csv"))
    if cached and not refresh:
        return pd.read_csv(cached[-1], parse_dates=["date"])
    reply = requests.get("https://fcainfoweb.nic.in/Default.aspx", timeout=90,
                         headers={"User-Agent":
                                  "python-for-indian-economics (teaching use)"})
    reply.raise_for_status()
    page = reply.text
    day = re.search(r'id="lblDate1">([\d/]+)<', page).group(1)
    names = re.findall(r'id="GridViewRetail\w+?_lblCommName">([^<]+)<', page)
    prices = re.findall(r'id="GridViewRetail\w+?_lblPrices">([^<]+)<', page)
    df = pd.DataFrame({"commodity": [n.strip() for n in names],
                       "rs_per_kg": pd.to_numeric(prices, errors="coerce")})
    df["date"] = pd.to_datetime(day, format="%d/%m/%Y")
    df.to_csv(Path("data/raw") / f"doca_retail_prices_{df['date'][0]:%Y-%m-%d}.csv",
              index=False)
    return df


retail = doca_retail_prices()
print(f"{len(retail)} commodities, prices as on {retail['date'][0]:%d %B %Y}")
retail.head(12)

In [ ]:
# Nutrients per 100 g edible portion, from IFCT 2017 Table 1 (code in brackets).
# Energy is in kJ in IFCT; protein, fat and fibre in g. Milk fibre is
# blank in IFCT (below detectable limit), entered as 0.
IFCT = pd.DataFrame.from_records([
    # food,        IFCT code, DoCA name,        energy_kJ, protein, fat, fibre
    ("Rice",        "A015", "Rice",              1491, 7.94, 0.52, 2.81),
    ("Atta",        "A019", "Atta (Wheat)",      1340, 10.57, 1.53, 11.36),
    ("Tur dal",     "B021", "Tur/Arhar Dal",     1384, 21.70, 1.56, 9.06),
    ("Gram dal",    "B001", "Gram Dal",          1377, 21.55, 5.31, 15.15),
    ("Moong dal",   "B010", "Moong Dal",         1363, 23.88, 1.35, 9.37),
    ("Masoor dal",  "B013", "Masoor Dal",        1349, 24.35, 0.75, 10.43),
    ("Urad dal",    "B003", "Urad Dal",          1356, 23.06, 1.69, 11.93),
    ("Potato",      "F006", "Potato",            292, 1.54, 0.23, 1.71),
    ("Milk (cow)",  "L002", "Milk @",            305, 3.26, 4.48, 0.0),
], columns=["food", "ifct_code", "doca_name", "energy_kj", "protein_g",
            "fat_g", "fibre_g"]).set_index("food")
IFCT["energy_kcal"] = IFCT["energy_kj"] / 4.184          # 1 kcal = 4.184 kJ
IFCT["group"] = ["cereal"] * 2 + ["legume"] * 5 + ["vegetable", "milk"]

diet = IFCT.join(retail.set_index("commodity")["rs_per_kg"], on="doca_name")
diet["rs_per_100g"] = diet["rs_per_kg"] / 10
diet["kcal_per_rupee"] = diet["energy_kcal"] / diet["rs_per_100g"]
diet["protein_g_per_rupee"] = diet["protein_g"] / diet["rs_per_100g"]
diet[["ifct_code", "rs_per_kg", "energy_kcal", "protein_g",
      "kcal_per_rupee", "protein_g_per_rupee"]].round(2)

In [ ]:
from scipy.optimize import linprog

ENERGY_REQ = 2110.0      # kcal/day, sedentary man, ICMR-NIN 2020 Table 1b
PROTEIN_REQ = 54.0       # g/day, RDA, ICMR-NIN 2020 Table 2a

c = diet["rs_per_100g"].to_numpy()
A_ge = np.vstack([diet["energy_kcal"], diet["protein_g"]])
b_ge = np.array([ENERGY_REQ, PROTEIN_REQ])

lp1 = linprog(c, A_ub=-A_ge, b_ub=-b_ge, bounds=(0, None), method="highs")
print(lp1.message)
print(f"minimum cost: Rs {lp1.fun:.2f} per day")
pd.Series(100 * lp1.x, index=diet.index).loc[lambda s: s > 0].round(0)

In [ ]:
is_cereal = (diet["group"] == "cereal").to_numpy(dtype=float)
is_legume = (diet["group"] == "legume").to_numpy(dtype=float)
is_milk = (diet["group"] == "milk").to_numpy(dtype=float)
A_ratio = np.vstack([is_legume - is_cereal / 3,
                     is_milk - 2.5 / 3 * is_cereal])
A2 = np.vstack([A_ge, A_ratio])
b2 = np.concatenate([b_ge, [0.0, 0.0]])
ROWS = ["energy (kcal)", "protein (g)", "legume ratio", "milk ratio"]

lp2 = linprog(c, A_ub=-A2, b_ub=-b2, bounds=(0, None), method="highs")
print(lp2.message)
print(f"minimum cost: Rs {lp2.fun:.2f} per day "
      f"(Rs {30 * lp2.fun:,.0f} per 30-day month)")
plan = pd.DataFrame({"grams per day": 100 * lp2.x,
                     "Rs per day": c * lp2.x}, index=diet.index)
plan = plan[plan["grams per day"] > 0.5]
print(plan.round(1))
achieved = A2 @ lp2.x
pd.DataFrame({"achieved": achieved, "required": b2}, index=ROWS).round(1)

## 26.5 Duality and shadow prices

In [ ]:
shadow = pd.Series(-lp2.ineqlin.marginals, index=ROWS)
print("shadow prices (Rs per unit of the requirement):")
print(shadow.round(5).to_string())
print(f"\ndual objective b'y = {b2 @ shadow:.4f}   primal cost c'x = {lp2.fun:.4f}")

# Check by re-solving with 100 extra kcal
b_more = b2.copy()
b_more[0] += 100
lp3 = linprog(c, A_ub=-A2, b_ub=-b_more, bounds=(0, None), method="highs")
print(f"cost of 100 more kcal: Rs {lp3.fun - lp2.fun:.4f} "
      f"(shadow price x 100 = Rs {100 * shadow.iloc[0]:.4f})")

In [ ]:
energy_grid = np.arange(1500, 3801, 50)
costs = []
for e in energy_grid:
    b_e = b2.copy()
    b_e[0] = e
    costs.append(linprog(c, A_ub=-A2, b_ub=-b_e, bounds=(0, None),
                         method="highs").fun)
costs = np.array(costs)
slopes = np.diff(costs) / np.diff(energy_grid)
print(f"slope of cost in energy: min Rs {slopes.min():.5f}, "
      f"max Rs {slopes.max():.5f} per kcal")
for e in (1660, 2110, 2710, 3470):
    b_e = b2.copy()
    b_e[0] = e
    print(f"  {e} kcal: Rs {linprog(c, A_ub=-A2, b_ub=-b_e, bounds=(0, None), method='highs').fun:6.2f} per day")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(energy_grid, costs, color="#1f4e79", linewidth=2)
for e, label in ((1660, "woman,\nsedentary"), (2110, "man,\nsedentary"),
                 (2710, "man,\nmoderate"), (3470, "man,\nheavy work")):
    ax.axvline(e, color="#9db7d5", linestyle=":", linewidth=1)
    ax.text(e + 20, costs.min(), label, fontsize=8, color="grey", va="bottom")
ax.set_xlabel("energy requirement, kcal per day")
ax.set_ylabel("minimum cost, Rs per day")
ax.set_title("Cost of the cheapest balanced diet rises linearly with energy",
             loc="left", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.06, 1, 1))
fig.text(0.01, 0.005, "Prices: DoCA all-India average retail prices "
         f"({retail['date'][0]:%d %b %Y}). Nutrients: IFCT 2017, ICMR-NIN. "
         "Requirements and 3:1:2.5 composition: ICMR-NIN (2020).\n"
         "Protein >= 54 g/day throughout. Nine foods only; no vegetables "
         "other than potato, no oil, fruit or micronutrients.",
         fontsize=8, color="grey")
plt.show()

## 26.6 Maximum likelihood: the distribution of salaries

In [ ]:
from indiadata import read_plfs, labour_status

p = read_plfs("perv1", columns=["ern_reg", "acws", "age", "sex"])
sal = p[(p["acws"] == 31) & (p["ern_reg"] > 0) & p["age"].between(15, 59)]
y = sal["ern_reg"].to_numpy(dtype=float)
w = sal["weight"].to_numpy(dtype=float)
w = w / w.mean()                      # normalise: weights now average 1
print(f"salaried workers aged 15-59 with earnings: {len(y):,}")

logy = np.log(y)


def negloglik(theta):
    mu, log_sigma = theta              # optimise log(sigma) so sigma > 0
    sigma = np.exp(log_sigma)
    z = (logy - mu) / sigma
    return -np.sum(w * (-logy - log_sigma - 0.5 * np.log(2 * np.pi) - 0.5 * z ** 2))


mle = optimize.minimize(negloglik, x0=[np.log(np.median(y)), 0.0],
                        method="BFGS")
mu_hat, sigma_hat = mle.x[0], np.exp(mle.x[1])
print(f"MLE: mu = {mu_hat:.4f}, sigma = {sigma_hat:.4f} "
      f"({mle.nit} iterations, converged = {mle.success})")

# The log-normal MLE has a closed form: weighted mean and s.d. of log y
mu_cf = np.average(logy, weights=w)
sigma_cf = np.sqrt(np.average((logy - mu_cf) ** 2, weights=w))
print(f"closed form: mu = {mu_cf:.4f}, sigma = {sigma_cf:.4f}")

In [ ]:
from scipy import stats


def weighted_quantile(x, wt, q):
    order = np.argsort(x)
    cum = np.cumsum(wt[order]) / wt.sum()
    return x[order][np.searchsorted(cum, q)]


fit = stats.lognorm(s=sigma_hat, scale=np.exp(mu_hat))
check = pd.DataFrame({
    "data (weighted)": [weighted_quantile(y, w, 0.5), np.average(y, weights=w),
                        weighted_quantile(y, w, 0.9), weighted_quantile(y, w, 0.99)],
    "log-normal fit": [fit.median(), fit.mean(), fit.ppf(0.9), fit.ppf(0.99)]},
    index=["median", "mean", "90th percentile", "99th percentile"])
check.round(0)

In [ ]:
se = np.sqrt(np.diag(mle.hess_inv))
print(f"approx. s.e. of mu: {se[0]:.4f}; of log sigma: {se[1]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
bins = np.linspace(np.log(500), np.log(300_000), 60)
ax.hist(logy, bins=bins, weights=w, density=True, color="#9db7d5",
        label="PLFS 2023-24, weighted")
xs = np.linspace(bins[0], bins[-1], 400)
ax.plot(xs, stats.norm.pdf(xs, mu_hat, sigma_hat), color="#c55a11",
        linewidth=2, label="log-normal fitted by maximum likelihood")
ticks = [1_000, 3_000, 10_000, 30_000, 100_000, 300_000]
ax.set_xticks(np.log(ticks), [f"{t:,}" for t in ticks])
ax.set_xlabel("monthly earnings, Rs (log scale)")
ax.set_ylabel("density of log earnings")
ax.set_title("Salaries of regular wage workers are roughly log-normal",
             loc="left", fontsize=12)
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "Regular wage/salaried workers (current weekly status "
         "31) aged 15-59, earnings in the previous month. Source: NSO, PLFS "
         "2023-24 unit-level data (perv1).", fontsize=8, color="grey")
plt.show()

### A probit by maximum likelihood

In [ ]:
import statsmodels.api as sm

adults = p[p["age"].between(15, 59) & p["sex"].isin([1, 2])]
d = (labour_status(adults["acws"], weekly=True) != "out of LF").to_numpy(float)
age = adults["age"].to_numpy(float)
X = np.column_stack([np.ones(len(d)), (adults["sex"] == 2).to_numpy(float),
                     (age - 35) / 10, ((age - 35) / 10) ** 2])
labels = ["const", "female", "age (decades from 35)", "age squared"]
del p, sal                                     # free memory


def probit_negll(beta):
    xb = X @ beta
    return -(d * stats.norm.logcdf(xb) + (1 - d) * stats.norm.logsf(xb)).sum()


def probit_grad(beta):
    xb = X @ beta
    # d/dxb of log Phi = phi/Phi (inverse Mills ratio); computed stably
    lam1 = np.exp(stats.norm.logpdf(xb) - stats.norm.logcdf(xb))
    lam0 = np.exp(stats.norm.logpdf(xb) - stats.norm.logsf(xb))
    return -X.T @ (d * lam1 - (1 - d) * lam0)


by_hand = optimize.minimize(probit_negll, x0=np.zeros(X.shape[1]),
                            jac=probit_grad, method="BFGS")
sm_fit = sm.Probit(d, X).fit(disp=False)
pd.DataFrame({"by hand (BFGS)": by_hand.x, "statsmodels": sm_fit.params,
              "s.e. (statsmodels)": sm_fit.bse}, index=labels).round(4)

In [ ]:
print(f"persons: {len(d):,} | log likelihood: by hand {-by_hand.fun:,.2f}, "
      f"statsmodels {sm_fit.llf:,.2f}")
xb_mean = X.mean(axis=0) @ by_hand.x
print(f"marginal effect of being female at the mean: "
      f"{stats.norm.pdf(xb_mean) * by_hand.x[1]:.3f}")

## 26.7 Answering the question

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading